# Maternal LFP Data Preprocessing

This notebook builds the **9 spectral feature pkls** consumed by the six
downstream Electome Factor (EF) task notebooks. It calls helpers from:

- `src/lfp_features.py` -- Power + Coherence (raw .mat -> per-mouse pkl)
- `src/dataset_assembly.py` -- behavior labels + aggregation + filter/trim

## Pipeline

| # | Section | Inputs | Produces |
|---|---------|--------|----------|
| 1 | Setup | paths, configs, mouse list | (config) |
| 2 | 3-band Welch | raw `.mat` LFP files | per-mouse 3-band pkls |
| 3 | Behavior labels | xlsx files | adds label fields to per-mouse pkls |
| 4 | 3-band aggregate + trim | per-mouse pkls | **pkl #1, #2, #4** |
| 5 | Behavior aggregate + Sara align | labeled per-mouse pkls | **pkl #7, #8, #9, #10** |
| 6 | 1Hz parallel pipeline | raw `.mat` LFP files | **pkl #5, #6** |

## 9 downstream pkls (pkl #3 superseded by pkl #4, see CLEANUP_LOG)

1. `full_spec_features_8roi.pkl` -- 3-band stage features, 14 mice
2. `full_spec_features_8roi_Trim_All.pkl` -- same, trimmed
4. `full_onnest_spec_features_8roi_Jan212026_Trim.pkl` -- onnest 3-band P1/3/8/14
5. `full_spec_features_Trim_All.pkl` -- 1Hz stage features, trimmed
6. `full_onnest_spec_features_Trim.pkl` -- 1Hz onnest, trimmed
7. `full_onnest_lick_Trim.pkl` -- P3 onnest + licking
8. `full_all_behaviors_no_nursing_field_Trim.pkl` -- P3 all behaviors (no nursing)
9. `full_all_behaviors_complete_trimmed.pkl` -- P3 with nursing aligned (Sara Jan 6 request)
10. `P4_pup_retrieval_detail.pkl` -- P4 pup retrieval 4-level labels

## Conventions

- **8 ROIs**: PrL, IL, Nac, MeA, CeA, BLA, Vhipp, VTA
- **Window**: 3.0 s (both 3-band and 1Hz pipelines)
- **Cohort**: 14 mice (8 C-group control + 6 E-group ELS)


# 1. Setup

In [2]:
# Allow imports from ../src
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if os.path.join(_repo_root, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(_repo_root, 'src'))

import pickle
import numpy as np
import pandas as pd
# lpne is a Carlson Lab library, GitHub-only (not on PyPI).
# Auto-install on first run if missing. To install manually:
#   pip install git+https://github.com/carlson-lab/lpne.git
try:
    import lpne
except ImportError:
    import subprocess
    print("Installing lpne from GitHub (first-time setup)...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'git+https://github.com/carlson-lab/lpne.git'])
    import lpne

import lfp_features as feat
from dataset_assembly import (
    get_filenames, canonical_id, strip_mouse_prefix,
    add_onnest_labels_to_pkls, add_behavior_labels_to_pkls,
    add_pup_retrieval_detail_to_pkls,
    aggregate_per_mouse_pkls, filter_and_trim_data,
    align_nursing_complete,
)


Installing lpne from GitHub (first-time setup)...
  Cloning https://github.com/carlson-lab/lpne.git to /private/var/folders/g4/n3wgfscs0cz3llsz9vwf6bj00000gn/T/pip-req-build-95yvmljz


  Running command git clone -q https://github.com/carlson-lab/lpne.git /private/var/folders/g4/n3wgfscs0cz3llsz9vwf6bj00000gn/T/pip-req-build-95yvmljz


  Resolved https://github.com/carlson-lab/lpne.git to commit 5d1e31b6689c0575ef2e617d5452dbf9822b98b4
  Created wheel for lpne: filename=lpne-0.1.24-py3-none-any.whl size=90546 sha256=ae8a8174c7484afdd6520e34880890d7d07ac7be99e7401b9e76ab82b1276266
  Stored in directory: /private/var/folders/g4/n3wgfscs0cz3llsz9vwf6bj00000gn/T/pip-ephem-wheel-cache-4uq17u0u/wheels/23/ac/36/04fa21cb215c617a6c2be1b4a7caa5ffebc78afcd5e13bdfed
Successfully built lpne


In [3]:
# ============================================================
# Pipeline configuration (single source of truth)
# ============================================================
FS = 1000                       # original sampling rate (Hz)
WINDOW_DURATION = 3.0           # seconds per analysis window (both pipelines)

# 14 target mice (8 control + 6 ELS), all with 8 ROIs histologically verified
TARGET_MICE_FULL = [
    "MouseC1F3ELS32", "MouseC2F4ELS18", "MouseC5F3ELS40", "MouseC5F4ELS20",
    "MouseC6F5ELS42", "MouseC7ELS11",   "MouseC7F1ELS22", "MouseC7F2ELS45",
    "MouseE1F4ELS33", "MouseE2ELS3",    "MouseE3F1ELS37", "MouseE4F2ELS39",
    "MouseE5F1ELS41", "MouseE6F4ELS44",
]
TARGET_MICE_SHORT = [
    "C1_ELS32", "C2_ELS18", "C5_ELS40", "C5_ELS20",
    "C6_ELS42", "C7_ELS11", "C7_ELS22", "C7_ELS45",
    "E1_ELS33", "E2_ELS3",  "E3_ELS37", "E4_ELS39",
    "E5_ELS41", "E6_ELS44",
]
# Prefix-stripped form, used by the 3-band stage trim:
TARGET_MICE_NOPREFIX = [m.replace("Mouse", "") for m in TARGET_MICE_FULL]

# 3-band Welch configuration (delta-theta / alpha / low-beta bands)
CONFIG_3BAND = dict(
    min_freq=1, max_freq=70,
    freq_bands=[(2, 7), (8, 12), (14, 23)],
    new_fs=100, nperseg=200,
    band_upper_inclusive=True, clip_for_safety=False,
)

# 1Hz Welch configuration (54 narrow 1-Hz bins from 2-56 Hz)
CONFIG_1HZ = dict(
    min_freq=2, max_freq=57,
    freq_bands=[(i, i+1) for i in range(2, 56)],
    new_fs=200, nperseg=400,
    band_upper_inclusive=False, clip_for_safety=True,
)

# Trim criteria (samples per recording, with 3 s windows)
SAMPLES_PER_MIN = 20    # 60 s / 3 s
SAMPLES_PER_HOUR = SAMPLES_PER_MIN * 60
TRIM_STAGE = {
    "P1":       4 * SAMPLES_PER_HOUR,
    "P3":       3 * SAMPLES_PER_HOUR,
    "P8":       2 * SAMPLES_PER_HOUR,
    "P14":      1 * SAMPLES_PER_HOUR,
    "P4 home":  20 * SAMPLES_PER_MIN,
    "Pre home": 10 * SAMPLES_PER_MIN,
    "Ges":      10 * SAMPLES_PER_MIN,
    "P20":      10 * SAMPLES_PER_MIN,
}
TRIM_ONNEST = {k: TRIM_STAGE[k] for k in ("P1", "P3", "P8", "P14")}
TRIM_P3_ONLY = {"P3": 3 * SAMPLES_PER_HOUR}

print(f"WINDOW_DURATION = {WINDOW_DURATION}")
print(f"Target mice: {len(TARGET_MICE_FULL)}")


WINDOW_DURATION = 3.0
Target mice: 14


In [ ]:
# ============================================================
# Data paths -- configure DATA_ROOT to match your RDSS mount.
# ============================================================
DATA_ROOT = "/Volumes/rdss_rhultman/datashare/ELS and Maternal Behavior"

# Raw LFP directories (one per recording stage)
exp_dir_P1   = f"{DATA_ROOT}/P1 Recordings"
exp_dir_P3   = f"{DATA_ROOT}/P3 Recordings"
exp_dir_P4   = f"{DATA_ROOT}/P4 Recordings"
exp_dir_P8   = f"{DATA_ROOT}/P8 Recordings"
exp_dir_P14  = f"{DATA_ROOT}/P14 Recordings"
exp_dir_P20  = f"{DATA_ROOT}/P20 Recordings"
exp_dir_Ges  = f"{DATA_ROOT}/Gestational Recordings"
exp_dir_Pre  = f"{DATA_ROOT}/Pre-conception Recordings"

STAGE_DIRS_3BAND = [
    ("Pre home", f"{exp_dir_Pre}/Home cage/Data"),
    ("Ges",      f"{exp_dir_Ges}/Data"),
    ("P1",       f"{exp_dir_P1}/Data"),
    ("P3",       f"{exp_dir_P3}/Data"),
    ("P4 home",  f"{exp_dir_P4}/Home cage/Data"),
    ("P8",       f"{exp_dir_P8}/Data"),
    ("P14",      f"{exp_dir_P14}/Data"),
    ("P20",      f"{exp_dir_P20}/Data"),
]
STAGE_DIRS_1HZ = STAGE_DIRS_3BAND   # same source files for both pipelines

# Per-mouse feature output dirs
FEATURE_DIR_3BAND = f"{DATA_ROOT}/Spec_Features_All"
FEATURE_DIR_8YES  = f"{DATA_ROOT}/Spec_Features_8Yes"
FEATURE_DIR_1HZ   = f"{DATA_ROOT}/Spec_Features_1Hz_8roi"

# Final output pkls (the 9 downstream files)
PKL_STAGE_3B          = f"{FEATURE_DIR_8YES}/combine/full_spec_features_8roi.pkl"           # #1
PKL_STAGE_3B_TRIM     = f"{FEATURE_DIR_8YES}/combine/full_spec_features_8roi_Trim_All.pkl"  # #2
PKL_ONNEST_3B_TRIM    = f"{FEATURE_DIR_8YES}/combined/full_onnest_spec_features_8roi_Jan212026_Trim.pkl"   # #4
PKL_STAGE_1HZ         = f"{FEATURE_DIR_1HZ}/combined/full_spec_features.pkl"
PKL_STAGE_1HZ_TRIM    = f"{FEATURE_DIR_1HZ}/combined/full_spec_features_Trim_All.pkl"       # #5
PKL_ONNEST_1HZ        = f"{FEATURE_DIR_1HZ}/combined/full_onnest_spec_features.pkl"
PKL_ONNEST_1HZ_TRIM   = f"{FEATURE_DIR_1HZ}/combined/full_onnest_spec_features_Trim.pkl"    # #6
PKL_LICK              = f"{DATA_ROOT}/Spec_Features_All/Combined/P3_onnest_lick/full_onnest_lick.pkl"
PKL_LICK_TRIM         = f"{DATA_ROOT}/Spec_Features_All/Combined/P3_onnest_lick/full_onnest_lick_Trim.pkl"  # #7
PKL_BEHAV_NONURSING   = f"{DATA_ROOT}/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field.pkl"
PKL_BEHAV_NONURSING_TRIM = f"{DATA_ROOT}/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_no_nursing_field_Trim.pkl"  # #8
PKL_BEHAV_NURSING     = f"{DATA_ROOT}/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_with_nursing.pkl"
PKL_BEHAV_COMPLETE    = f"{DATA_ROOT}/Spec_Features_All/Combined/P3_onnest_lick_selfgroom_nurse/full_all_behaviors_complete_trimmed.pkl"  # #9
PKL_PUP_DETAIL        = f"{DATA_ROOT}/Spec_Features_All/Combined/P4_pup_retrieval_detail.pkl"   # #10

# Intermediate paths
PKL_ONNEST_INTERMEDIATE = f"{DATA_ROOT}/Spec_Features_All/Combined/full_onnest_spec_features.pkl"


# 2. 3-band Feature Extraction (Power + Coherence)

Computes Welch power and squared coherence for 8 ROIs across 3 wide frequency
bands `(2-7)`, `(8-12)`, `(14-23)` Hz on 3-second sliding windows.

Calls `lfp_features.extract_features_for_stage` for each recording stage.
The output is per-mouse pkls in `Spec_Features_All/`. Each pkl contains
`power`, `coh_sq_coherence`, `freq_band`, `region`, `region_pair`, `mouse_id`,
`period`, after log10 + per-file min-max normalization.


In [ ]:
# Run 3-band Welch extraction for all stages
for stage_name, stage_dir in STAGE_DIRS_3BAND:
    lfp_fns = get_filenames(stage_dir, "_LFP.mat")
    chans_fns = get_filenames(stage_dir, "_CHANS.mat")
    print(f"\n=== {stage_name}: {len(lfp_fns)} files ===")
    feat.extract_features_for_stage(
        lfp_fns, chans_fns, stage_name, FEATURE_DIR_3BAND,
        fs=FS, window_duration=WINDOW_DURATION,
        **CONFIG_3BAND,
    )


# 3. Behavior Labels

Adds behavior label fields to per-mouse pkls **in place**.

- `onnest_label` (binary, P1/P3/P8/P14): >= half-window overlap with on-nest bouts
- `onnest_raw`, `licking`, `selfgroom`, `nursing` (P3 only): > half-window overlap (majority)
- `pup_retrieval_detail` (4-level, P4 home only): 0=no trial / 1=trial no retrieval / 3=partial / 4=successful


In [ ]:
# Behavior xlsx file lists
P1_onnest_fns = get_filenames(f"{exp_dir_P1}/Behavioral Scoring/on-nest", "xlsx")
P3_onnest_fns = get_filenames(f"{exp_dir_P3}/Behavioral Scoring/maternal behaviors/on-nest/Excel", "xlsx")
P8_onnest_fns = get_filenames(f"{exp_dir_P8}/Behavioral Scoring/on-nest", "xlsx")

P3_nurse_fns     = get_filenames(f"{exp_dir_P3}/Behavioral Scoring/maternal behaviors/nursing", "xlsx")
P3_lick_fns      = get_filenames(f"{exp_dir_P3}/Behavioral Scoring/maternal behaviors/licking_grooming (of pups)", "xlsx")
P3_selfgroom_fns = get_filenames(f"{exp_dir_P3}/Behavioral Scoring/non-maternal behaviors/grooming (of self)", "xlsx")

P4_pup_fns = get_filenames(f"{exp_dir_P4}/Home cage pup retrieval/Scoring of Pup Retrieval Behavior", ".xlsx")
P4_pup_trial_excel = f"{exp_dir_P4}/Home cage pup retrieval/Scoring of Pup Retrieval Behavior/P4_Pup_Retrieval_Trial_Times_and_Notes.xlsx"

# Per-mouse pkl file lists (produced by Section 2)
all_fns_3band = get_filenames(FEATURE_DIR_3BAND, "pkl")
p1_fns  = [fn for fn in all_fns_3band if fn.endswith("_P1.pkl")]
p3_fns  = [fn for fn in all_fns_3band if fn.endswith("_P3.pkl")]
p8_fns  = [fn for fn in all_fns_3band if fn.endswith("_P8.pkl")]
p14_fns = [fn for fn in all_fns_3band if fn.endswith("_P14.pkl")]
p4_home_fns = [fn for fn in all_fns_3band if fn.endswith("_P4 home.pkl")]


In [ ]:
# Add binary onnest_label to P1/P3/P8 (P14 has no on-nest scoring)
print("Adding onnest_label to P1/P3/P8 pkls...")
add_onnest_labels_to_pkls(p1_fns, P1_onnest_fns, WINDOW_DURATION)
add_onnest_labels_to_pkls(p3_fns, P3_onnest_fns, WINDOW_DURATION)
add_onnest_labels_to_pkls(p8_fns, P8_onnest_fns, WINDOW_DURATION)


In [ ]:
# Add P3-specific behavior labels: onnest_raw, licking, selfgroom, nursing
print("Adding P3 behavior labels...")
p3_updated_fns = add_behavior_labels_to_pkls(p3_fns, {
    "onnest_raw":  P3_onnest_fns,
    "licking":     P3_lick_fns,
    "selfgroom":   P3_selfgroom_fns,
    "nursing":     P3_nurse_fns,
}, WINDOW_DURATION)


In [ ]:
# Add 4-level pup_retrieval_detail to P4 home pkls
print("Adding pup_retrieval_detail to P4 home pkls...")
p4_updated_fns = add_pup_retrieval_detail_to_pkls(
    p4_home_fns, P4_pup_fns, P4_pup_trial_excel, WINDOW_DURATION)


# 4. 3-band Aggregate + Filter + Trim

Selects target mice (14, all 8 ROIs verified), aggregates per-mouse pkls,
and trims to canonical recording lengths.

**Produces pkl #1, #2, #4.** (pkl #3 was an older onnest variant superseded
by pkl #4; see CLEANUP_LOG.md.)


In [ ]:
# Copy per-mouse pkls of the 14 target mice to Spec_Features_8Yes/
# (excluding Pre pup and P4 open stages, which are not used in EF tasks)
import shutil
from pathlib import Path

os.makedirs(FEATURE_DIR_8YES, exist_ok=True)
EXCLUDE_STAGES = ["Pre pup", "P4 open"]

copied = 0
for fp in all_fns_3band:
    basename = Path(fp).stem
    if any(stage in basename for stage in EXCLUDE_STAGES):
        continue
    if canonical_id(basename) not in TARGET_MICE_SHORT:
        continue
    shutil.copy2(fp, os.path.join(FEATURE_DIR_8YES, Path(fp).name))
    copied += 1
print(f"Copied {copied} pkls to {FEATURE_DIR_8YES}/")


In [ ]:
# Aggregate 3-band stage features -> pkl #1
all_files_8yes = get_filenames(FEATURE_DIR_8YES, ".pkl")
aggregate_per_mouse_pkls(all_files_8yes, PKL_STAGE_3B, label_keys=())

# Trim 3-band stage -> pkl #2
filter_and_trim_data(
    input_path=PKL_STAGE_3B, output_path=PKL_STAGE_3B_TRIM,
    mice_to_keep=TARGET_MICE_NOPREFIX,
    trim_criteria=TRIM_STAGE, id_transform=strip_mouse_prefix,
)


In [ ]:
# Aggregate P1+P3+P8+P14 onnest pkls (with Jan 12 E5ELS41 P1 fix already applied
# during Section 3 add_onnest_labels_to_pkls re-run, and Jan 12 P14 added below).
onnest_fns_all = p1_fns + p3_fns + p8_fns + p14_fns
aggregate_per_mouse_pkls(
    onnest_fns_all, PKL_ONNEST_INTERMEDIATE,
    label_keys=("onnest_label",),
)

# Filter to 14 mice in canonical short form, trim each recording -> pkl #4
filter_and_trim_data(
    input_path=PKL_ONNEST_INTERMEDIATE, output_path=PKL_ONNEST_3B_TRIM,
    mice_to_keep=TARGET_MICE_SHORT,
    trim_criteria=TRIM_ONNEST, id_transform=canonical_id,
)


# 5. Behavior Aggregate + Trim + Sara Align

Aggregates the P3 behavior-labeled pkls and P4 pup-retrieval pkls into the
analysis-ready combined datasets.

**Produces pkl #7, #8, #9, #10.**


In [ ]:
# pkl #7: P3 onnest + licking -> aggregate -> trim
aggregate_per_mouse_pkls(p3_updated_fns, PKL_LICK,
                          label_keys=("onnest_raw", "licking"))
filter_and_trim_data(
    input_path=PKL_LICK, output_path=PKL_LICK_TRIM,
    mice_to_keep=TARGET_MICE_FULL,
    trim_criteria=TRIM_P3_ONLY,
)


In [ ]:
# pkl #8: P3 all behaviors WITHOUT nursing field -> aggregate -> trim
aggregate_per_mouse_pkls(
    p3_updated_fns, PKL_BEHAV_NONURSING,
    label_keys=("onnest_raw", "licking", "selfgroom"),
)
filter_and_trim_data(
    input_path=PKL_BEHAV_NONURSING, output_path=PKL_BEHAV_NONURSING_TRIM,
    mice_to_keep=TARGET_MICE_FULL,
    trim_criteria=TRIM_P3_ONLY,
)

# Intermediate: aggregate ONLY the mice that have nursing data (subset of p3_updated_fns)
p3_with_nursing_fns = []
for fp in p3_updated_fns:
    with open(fp, "rb") as f:
        d = pickle.load(f)
    if "nursing" in d:
        p3_with_nursing_fns.append(fp)
print(f"P3 pkls with nursing field: {len(p3_with_nursing_fns)}")

aggregate_per_mouse_pkls(
    p3_with_nursing_fns, PKL_BEHAV_NURSING,
    label_keys=("onnest_raw", "licking", "selfgroom", "nursing"),
)


In [ ]:
# pkl #9: Sara Jan 6 -- align nursing field, filter, trim to 3600 per recording
align_nursing_complete(
    with_nursing_path=PKL_BEHAV_NURSING,
    no_nursing_path=PKL_BEHAV_NONURSING,
    output_path=PKL_BEHAV_COMPLETE,
    target_mice=TARGET_MICE_FULL,
    max_samples=3600,
)


In [ ]:
# pkl #10: P4 home pup retrieval detail -- aggregate (no trim)
aggregate_per_mouse_pkls(
    p4_updated_fns, PKL_PUP_DETAIL,
    label_keys=("pup_retrieval_detail",),
)


# 6. 1Hz Parallel Pipeline

Same Power + Coherence as Section 2 but with 54 narrow 1-Hz frequency bins
(2-56 Hz) and higher decimation rate / longer Welch segments. **Window
duration is still 3.0 s** (same as 3-band); the "1Hz" name refers to
frequency resolution, not time resolution.

**Produces pkl #5, #6.**


In [ ]:
# Run 1Hz Welch extraction for all stages
for stage_name, stage_dir in STAGE_DIRS_1HZ:
    lfp_fns = get_filenames(stage_dir, "_LFP.mat")
    chans_fns = get_filenames(stage_dir, "_CHANS.mat")
    print(f"\n=== 1Hz: {stage_name} ({len(lfp_fns)} files) ===")
    feat.extract_features_for_stage(
        lfp_fns, chans_fns, stage_name, FEATURE_DIR_1HZ,
        fs=FS, window_duration=WINDOW_DURATION,
        **CONFIG_1HZ,
    )


In [ ]:
# Add onnest_label to 1Hz P1/P3/P8 per-mouse pkls (same window_duration as 3-band)
all_fns_1hz = get_filenames(FEATURE_DIR_1HZ, "pkl")
p1_fns_1hz  = [fn for fn in all_fns_1hz if fn.endswith("_P1.pkl")]
p3_fns_1hz  = [fn for fn in all_fns_1hz if fn.endswith("_P3.pkl")]
p8_fns_1hz  = [fn for fn in all_fns_1hz if fn.endswith("_P8.pkl")]

print("Adding 1Hz onnest_label to P1/P3/P8 pkls...")
add_onnest_labels_to_pkls(p1_fns_1hz, P1_onnest_fns, WINDOW_DURATION)
add_onnest_labels_to_pkls(p3_fns_1hz, P3_onnest_fns, WINDOW_DURATION)
add_onnest_labels_to_pkls(p8_fns_1hz, P8_onnest_fns, WINDOW_DURATION)


In [ ]:
# pkl #5: aggregate 1Hz stage features -> trim
aggregate_per_mouse_pkls(all_fns_1hz, PKL_STAGE_1HZ, label_keys=())
filter_and_trim_data(
    input_path=PKL_STAGE_1HZ, output_path=PKL_STAGE_1HZ_TRIM,
    mice_to_keep=TARGET_MICE_NOPREFIX,
    trim_criteria=TRIM_STAGE, id_transform=strip_mouse_prefix,
)


In [ ]:
# pkl #6: aggregate 1Hz onnest pkls -> trim
onnest_fns_1hz = p1_fns_1hz + p3_fns_1hz + p8_fns_1hz
aggregate_per_mouse_pkls(
    onnest_fns_1hz, PKL_ONNEST_1HZ,
    label_keys=("onnest_label",),
)
filter_and_trim_data(
    input_path=PKL_ONNEST_1HZ, output_path=PKL_ONNEST_1HZ_TRIM,
    mice_to_keep=TARGET_MICE_FULL,
    trim_criteria={k: TRIM_ONNEST[k] for k in ("P1", "P3", "P8")},
)
